In [ ]:
# Clone the project repo (Colab setup)
import os
if not os.path.exists("/content/WIA1006_MLPROJECT"):
    !git clone -b aaron-staging https://github.com/aarntn/WIA1006_MLPROJECT.git /content/WIA1006_MLPROJECT
else:
    !cd /content/WIA1006_MLPROJECT && git pull
%cd /content/WIA1006_MLPROJECT

In [ ]:
# Install dependencies
!pip install -q xgboost hdbscan umap-learn shap flaml autogluon.tabular statsmodels

# Swipe Atlas Final Workflow

Main project story: predict dating-app engagement (`mutual_matches`) and segment users by behavior. `match_outcome` is retained as a no-signal case study because the synthetic labels are balanced and not practically predictable from the available features.

In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = Path.cwd()
print(ROOT)

## 1. EDA and Signal Check

In [ ]:
subprocess.run([sys.executable, "scripts/01_eda.py"], check=True)
subprocess.run([sys.executable, "scripts/02_signal_test.py"], check=True)

In [ ]:
from IPython.display import Image, display
from pathlib import Path
fig = Path("reports/figures")
display(Image(str(fig / "01_target_distribution.png")))
display(Image(str(fig / "07_pca_structure.png")))
display(Image(str(fig / "10_signal_summary.png")))
display(Image(str(fig / "11_confusion_matrix_3class.png")))

## 2. Engagement Modeling

The official model predicts `mutual_matches` without using `likes_received` or `match_outcome`. A paired-feature comparison is generated separately to show leakage inflation.

In [ ]:
subprocess.run([sys.executable, "scripts/04_train_engagement_models.py"], check=True)

In [ ]:
from IPython.display import Image, display
from pathlib import Path
fig = Path("reports/figures")
display(Image(str(fig / "12_engagement_model_comparison.png")))
display(Image(str(fig / "13_leakage_comparison.png")))
display(Image(str(fig / "14b_learning_curve.png")))
display(Image(str(fig / "14c_shap_beeswarm.png")))

## 3. User Segmentation

In [ ]:
subprocess.run([sys.executable, "scripts/05_segmentation.py"], check=True)

In [ ]:
from IPython.display import Image, display
from pathlib import Path
fig = Path("reports/figures")
display(Image(str(fig / "15_kmeans_selection.png")))
display(Image(str(fig / "16_segment_umap.png")))
display(Image(str(fig / "17_segment_profiles.png")))

## 4. AutoML Comparison

Three AutoML frameworks were compared against the manually tuned model:

- **FLAML** (Microsoft) — Windows-native, executed locally
- **AutoGluon Tabular** — Windows-native, executed locally
- **auto-sklearn 2.0** (Feurer et al., JMLR 2022) — Linux/Colab-only; executed in Google Colab

All three land at R² ≈ 0, confirming that the near-zero signal is a dataset property, not a modelling failure.

In [ ]:
import pandas as pd
from IPython.display import Image, display
from pathlib import Path

results = pd.read_csv("reports/automl_results.csv")
display(results[["model","backend","status","r2","mae","rmse"]].round(4))

fig = Path("reports/figures")
display(Image(str(fig / "20_automl_comparison.png")))

## 5. Report Tables

In [ ]:
import pandas as pd
display(pd.read_csv("reports/engagement_model_results.csv"))
display(pd.read_csv("reports/segmentation_summary.csv"))

## 6. Conclusions

| Finding | Evidence |
|---------|----------|
|  has no predictive signal | All classifiers at 10% = random baseline; 0/23 tests survive Bonferroni correction |
|  also near-zero signal | Tuned model R² = −0.001 on safe feature set |
|  causes leakage | Including it inflates R² from −0.001 to 0.127 |
| AutoML confirms the finding | FLAML R²=−0.006, AutoGluon R²=0.000, auto-sklearn R²=−0.000 |
| No cluster structure detected | Best silhouette = 0.019; HDBSCAN all-noise; GMM BIC monotone |
| Data is synthetic | Perfect class balance, near-uniform tag frequencies, zero missing values |

**Takeaway**: Honest ML reporting (near-zero R²) is more valuable than a fake high-accuracy model built on leaky features.